# 03 · Revised estimator: inference procedures, predictive families and censoring

This notebook answers the independent review of M0 (decision: MODIFY). Notebook 01 is kept unchanged as the record of the first validation, including its failures. Nothing here is meteorological evidence.

| review item | section |
|---|---|
| 1 separate evidence from effect size; dependence-aware inference; calibrate under misspecification, multimodal errors, serial dependence, sensor-correlated error | §1–§3, §6 |
| 2 flexible predictive density; binned vs continuous information | §4, §6 |
| 3 censored training likelihood | §5, §6 |
| 6 row-order dependence of the leakage audit | `policy.eta2` + `test_eta2_ignores_row_order_and_keeps_ties_together` |
| 7 new seeds and worlds as a separate validation suite | §6 (seeds ≥ 90 000, worlds not used in development) |

**Estimand discipline.** The held-out contrast estimates the information *captured by the fitted models*. It equals `I(D;V|C)` only when both models are correct (`E Δ̂ = I − KL_VC + KL_C`). Each table therefore compares estimates with exact synthetic truths. Where a procedure targets something else (the information of a binned D, an observation-level quantity under censoring, or the contrast of frozen models), that target is computed exactly too and named.

**Three questions, kept separate.** (Q0) *Is there any information?* A test of `H0: V ⊥ D | C` at level 0.05. (Q1) *How much?* An interval for the captured information. (Q2) *Is it scientifically meaningful?* The interval compared with a threshold δ fixed before the data. A Q0 rejection is never read as Q2.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import replace

from joblib import Parallel, delayed
from scipy.stats import norm

from visual_hygrometry import data, inference, policy, scoring, splits, synthetic
from visual_hygrometry.synthetic import Node

plt.rcParams.update({'figure.dpi': 80, 'figure.figsize': (8, 3.2), 'axes.grid': True, 'grid.alpha': .3})
pd.set_option('display.precision', 4)
JOBS = 16
par = lambda: Parallel(n_jobs=JOBS, max_nbytes=None)

HOUR = ['hour_s1', 'hour_c1', 'hour_s2', 'hour_c2']
E2 = HOUR + ['temp_cond_c']
EDGES = [0.5, 1, 1.5, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]  # K, fixed before any run; finer near saturation
cyc = lambda a, b, peak=15: (lambda h: a + b * np.cos(2 * np.pi * (h - peak) / 24))


def gauss_world(gamma=.162, beta=.4, t_sd=3, r_sd=2, r_day=.5, r_ar=.9, v_day=.3, v_ar=.5):
    """Diurnal T, moisture residual r, D = mean(h) + beta T + r (D ~ 7 + 4 cos(hour) K), visual cue gamma r + noise."""
    return {'temp_cond_c': Node(sd=t_sd, mean=cyc(20, 5), day=.7, ar=.95), 'dres': Node(sd=r_sd, day=r_day, ar=r_ar),
            'd_ref_k': Node({'temp_cond_c': beta, 'dres': 1}, mean=cyc(7 - 20 * beta, 4 - 5 * beta)), 'vis_moist': Node({'dres': gamma}, 1.0, day=v_day, ar=v_ar)}


def rad_world(err_sd=1.0):
    """Shared radiation-biased unit: label D and conditioning T both carry err; the image sees the irradiance anomaly."""
    return {'t_true': Node(sd=3, mean=cyc(20, 5), day=.7, ar=.95), 'dres': Node(sd=2, day=.5, ar=.9),
            'd_true': Node({'t_true': .4, 'dres': 1}, mean=cyc(-1, 2)), 'cloud': Node(sd=1, day=.8, ar=.9),
            'err': Node({'cloud': err_sd}), 'temp_cond_c': Node({'t_true': 1, 'err': 1}), 'd_ref_k': Node({'d_true': 1, 'err': 1}),
            'vis_bright': Node({'cloud': 1}, .3)}


def regime(sds, p_humid=.5):
    """Day-level regime scales the moisture residual (SD sds[1] humid, sds[0] dry); mean unchanged."""
    def f(df, rng):
        r = rng.random(df.day.max() + 1)[df.day] < p_humid
        df['regime'] = r.astype(float)
        df['d_ref_k'] = df.d_ref_k - df.dres + np.where(r, sds[1], sds[0]) * df.dres / df.attrs['dres_sd']
        return df
    return f


def frame(W, n_days, seed):
    """Simulate a world, attach the fixed hour basis, apply its non-linear modification, label sensor ids."""
    df = synthetic.simulate(W['spec'], n_days, W.get('per_day', 48), W.get('hours', (6, 18)), seed=seed)
    df = pd.concat([df, data.fourier(df.hour, 24, 2, 'hour')], axis=1).assign(hour_local=df.hour, ref_sensor_id='ref', cond_sensor_id='ref' if W.get('shared') else 'cond')
    df.attrs['dres_sd'] = W['spec']['dres'].sd if 'dres' in W['spec'] else 1
    return W['modify'](df, np.random.default_rng(seed + 7)) if 'modify' in W else df


def iid(W):
    """Same marginal per-row distribution with independent rows (day = ar = 0): Monte-Carlo truths need no long-range averaging."""
    return W | {'spec': {k: replace(n, day=0.0, ar=0.0) for k, n in W['spec'].items()}}

## 1 · Procedures compared

| id | question | procedure | what it conditions on |
|---|---|---|---|
| `boot` | Q1 / Q0 | cross-fitted scores, percentile bootstrap over days (M0 baseline) | fitted models |
| `xt` | Q1 / Q0 | cross-fitted scores, day-cluster SE, t with G − 1 df | fitted models |
| `frz` | Q1 / Q0 | fit on half of the days (interleaved), freeze, score the other half once; day-cluster t | the frozen models; its target is their population contrast, computed on 200 extra days |
| `perm` | Q0 | refitting day-block permutation test: residuals of V given C (linear) permuted between whole days with rows matched by time of day, then re-projected on the orthogonal complement of C; the joint model is refitted for each of 39 draws | exchangeability of residual day-blocks |
| `diag` | adequacy | the same contrast with V replaced by all squares and pairwise products of the context columns; flag if its one-sided lower bound > 0 | — |

Tests are one-sided at α = 0.05 (bootstrap 5 % quantile, t₀.₉₅, or permutation p ≤ 0.05); intervals are two-sided 95 %.

In [2]:
def one_rep(W, seed, n_pop=200, n_perm=39):
    n = W['n_days']
    df = frame(W, n + n_pop, seed)
    Xc, Xvc, y, _ = policy.select_sci(df, W.get('level', 'e2'), W['ctx'], W['vis'], allow_shared_sensor=W.get('shared', False))
    C, X, y, g = Xc.to_numpy(), Xvc.to_numpy(), y.to_numpy(), df.day.to_numpy()
    s, slot = g < n, df.groupby('day').cumcount().to_numpy()
    sp = splits.kfold(splits.assign_folds(g[s], 5))
    r = scoring.contrast(C[s], X[s], y[s], sp, g[s])
    b = np.quantile(scoring.block_bootstrap(r.d, r.groups, 1000, seed), [.025, .05, .975])
    m, se, G = inference.cluster_se(r.d, r.groups)
    xt_lo, xt_hi = inference.cluster_interval(r.d, r.groups)
    half = splits.assign_folds(g[s], 2)
    fit = np.zeros(len(g), bool)
    fit[np.flatnonzero(s)[half == 0]] = True
    fz = scoring.contrast(C, X, y, [(fit, ~fit)], g)
    dz = np.full(len(g), np.nan)
    dz[fz.scored] = fz.d
    ev = s & ~fit
    fz_lo, fz_hi = inference.cluster_interval(dz[ev], g[ev])
    _, p, _ = inference.perm_test(C[s], X[s, C.shape[1]:], y[s], sp, g[s], slot[s], n_perm, seed + 500_000)  # permutation stream independent of the simulation
    Cz = (C - C.mean(0)) / C.std(0)
    iu = np.triu_indices(C.shape[1])
    rd = scoring.contrast(C[s], np.c_[C, Cz[:, iu[0]] * Cz[:, iu[1]]][s], y[s], sp, g[s])
    return dict(seed=seed, est=r.mean, boot_lo=b[0], boot_hi=b[2], boot_lo1=b[1], xt_lo=xt_lo, xt_hi=xt_hi, xt_lo1=inference.cluster_lower(r.d, r.groups),
                frz=np.mean(dz[ev]), frz_lo=fz_lo, frz_hi=fz_hi, frz_lo1=inference.cluster_lower(dz[ev], g[ev]), frz_target=np.nanmean(dz[~s]),
                p_perm=p, diag=inference.cluster_lower(rd.d, rd.groups) > 0)


def run(W, seeds): return pd.DataFrame(par()(delayed(one_rep)(W, s) for s in seeds))


def summarise(name, W, a):
    t = W['truth']
    return dict(world=name, truth=t, n_days=W['n_days'], reps=len(a), bias=a.est.mean() - t, sd_est=a.est.std(),
                rej_boot=(a.boot_lo1 > 0).mean(), rej_xt=(a.xt_lo1 > 0).mean(), rej_frz=(a.frz_lo1 > 0).mean(), rej_perm=(a.p_perm <= .05).mean(),
                cov_boot=((a.boot_lo <= t) & (a.boot_hi >= t)).mean(), cov_xt=((a.xt_lo <= t) & (a.xt_hi >= t)).mean(),
                cov_frz=((a.frz_lo <= t) & (a.frz_hi >= t)).mean(), cov_frz_target=((a.frz_lo <= a.frz_target) & (a.frz_hi >= a.frz_target)).mean(),
                frz_target_gap=(a.frz_target - t).mean(), diag_flag=a.diag.mean())

## 2 · Development suite: worlds

Same family as notebook 01, **new seeds** (2000+). Nulls cover the conditions the review names; `truth` is the information on the measured variables, `atm` the atmospheric information.

| world | condition | truth | atm |
|---|---|---|---|
| N1 | well-specified null, 60 days | 0 | 0 |
| N2 | strong serial dependence (residual 85 % day-level, AR 0.97; visual noise 60 % day-level) | 0 | 0 |
| N3a | context misspecified in hour (linear hour), sun-elevation cue | 0 | 0 |
| N3b | context misspecified in T: `D` has a 0.15 (T−20)² term, context linear in T, cue = (T−20)²/10 + noise | 0 | 0 |
| N4 | multimodal errors: day regimes with residual SD 0.7 / 2.5 K, uninformative cue | 0 | 0 |
| N5 | sensor-correlated error: one radiation-biased unit (1 K), brightness cue | 0.035 | 0 |
| P1, P2 | Gaussian positive, truth 0.05, 60 and 120 days | 0.05 | 0.05 |

In [3]:
def nonlinear_t(df, rng):
    df['d_ref_k'] = df.d_ref_k + 0.15 * (df.temp_cond_c - 20) ** 2
    df['vis_tq'] = (df.temp_cond_c - 20) ** 2 / 10 + rng.normal(0, .5, len(df))
    return df

sun = {'vis_sun': Node(sd=.05, mean=lambda h: np.sin(np.pi * (h - 6) / 12))}
DEV = {'N1': dict(spec=gauss_world(0), ctx=E2, vis=['vis_moist'], truth=0.0, n_days=60),
       'N2': dict(spec=gauss_world(0, r_day=.85, r_ar=.97, v_day=.6, v_ar=.9), ctx=E2, vis=['vis_moist'], truth=0.0, n_days=60),
       'N3a': dict(spec=gauss_world(0) | sun, ctx=['hour_local', 'temp_cond_c'], vis=['vis_sun'], truth=0.0, n_days=60),
       'N3b': dict(spec=gauss_world(0), ctx=E2, vis=['vis_tq'], truth=0.0, n_days=60, modify=nonlinear_t),
       'N4': dict(spec=gauss_world(0), ctx=E2, vis=['vis_moist'], truth=0.0, n_days=60, modify=regime((2.5, 0.7))),
       'N5': dict(spec=rad_world(1.0), ctx=E2, vis=['vis_bright'], truth=synthetic.cmi(rad_world(1.0), 'd_ref_k', 'vis_bright', 'temp_cond_c'), n_days=60, shared=True),
       'P1': dict(spec=gauss_world(.162), ctx=E2, vis=['vis_moist'], truth=synthetic.cmi(gauss_world(.162), 'd_ref_k', 'vis_moist', 'temp_cond_c'), n_days=60),
       'P2': dict(spec=gauss_world(.162), ctx=E2, vis=['vis_moist'], truth=synthetic.cmi(gauss_world(.162), 'd_ref_k', 'vis_moist', 'temp_cond_c'), n_days=120)}
dev_raw = {k: run(W, range(2000, 2200 if k in ('N1', 'N2', 'N4') else 2100)) for k, W in DEV.items()}
dev = pd.DataFrame([summarise(k, DEV[k], a) for k, a in dev_raw.items()]).set_index('world')
dev[['truth', 'n_days', 'bias', 'sd_est', 'rej_boot', 'rej_xt', 'rej_frz', 'rej_perm', 'diag_flag']]

,truth,n_days,bias,sd_est,rej_boot,rej_xt,rej_frz,rej_perm,diag_flag
world,,,,,,,,,
N1,0.0000,60,-0.0014,0.0047,0.005,0.005,0.040,0.075,0.005
N2,0.0000,60,-0.0049,0.0142,0.000,0.000,0.020,0.090,0.010
N3a,0.0000,60,0.0168,0.0082,0.540,0.540,0.410,0.000,0.310
N3b,0.0000,60,0.5773,0.1083,1.000,1.000,1.000,1.000,1.000
N4,0.0000,60,-0.0015,0.0045,0.010,0.010,0.035,0.060,0.000
N5,0.0353,60,-0.0022,0.0315,0.320,0.310,0.300,0.700,0.010
P1,0.0499,60,0.0010,0.0235,0.850,0.820,0.680,0.990,0.010
P2,0.0499,120,-0.0005,0.0125,1.000,1.000,0.870,1.000,0.010


Rejection rates are Type I errors in N1–N4 (Monte-Carlo SE at 0.05: 0.015 with 200 replicates for N1, N2, N4; 0.022 with 100 for N3a/b), power in P1–P2. In N5 they are the rate of rejecting an **atmospheric** null that is true, while the measured-variable null is false.

In [4]:
dev[['truth', 'cov_boot', 'cov_xt', 'cov_frz', 'cov_frz_target', 'frz_target_gap']]

,truth,cov_boot,cov_xt,cov_frz,cov_frz_target,frz_target_gap
world,,,,,,
N1,0.0000,0.83,0.855,0.850,0.910,-0.0046
N2,0.0000,0.81,0.865,0.855,0.915,-0.0152
N3a,0.0000,0.61,0.650,0.720,0.950,0.0161
N3b,0.0000,0.00,0.000,0.020,0.890,0.5864
N4,0.0000,0.86,0.910,0.875,0.915,-0.0040
N5,0.0353,0.85,0.850,0.870,0.880,-0.0075
P1,0.0499,0.83,0.860,0.930,0.950,-0.0042
P2,0.0499,0.94,0.960,0.980,0.940,-0.0013


## 3 · Reading the development suite

*Interpretation written after the development run; the resulting choices are in `docs/m0_decision_addendum.md`.*

- **Score-based intervals on cross-fitted scores (`boot`, `xt`)** are badly matched to both questions at 60 days. Under adequate-context nulls they almost never reject (0–1 %), because the null contrast is negatively biased (the joint model pays for a useless coefficient) and its SE shrinks with it. For a 0.05-nat effect they under-cover (0.83 / 0.86).
- **Frozen hold-out (`frz`)** has Type I 0.020–0.040 on the adequate-context nulls. Its interval covers its own target (the information the frozen models capture) in 0.88–0.95 of replicates in every world, misspecified ones included, and covers the truth in 0.93 at P1. The price is power (0.68 vs 0.82 at 60 days), because half the days fit and half evaluate.
- **Permutation (`perm`)** reaches power 0.99 at P1: its null distribution is the narrow spread of a null world (SD ≈ 0.005), not the wide sampling spread of a positive effect. Type I is 0.075 / 0.090 / 0.060 in N1 / N2 / N4 (SE 0.015), at or above nominal. Development runs outside the notebook (600 replicates, 19 draws) gave 0.040 for raw day-block permutation (exact, as theory requires) and 0.035 after residual re-projection, which is now in `inference.perm_test`. The excess in N2 with 39 draws is not explained and stays flagged.
- **N3a (hour-misspecified context):** slot-matched permutation preserves structure that depends only on time of day, so it never rejects; every score-interval test rejects in 41–54 % of replicates.
- **N3b (temperature-misspecified context):** every procedure rejects in 100 % of replicates, with a 0.58-nat estimate in a world with no information. No inference procedure separates "V repairs the context model" from "V carries information". Here only the adequacy diagnostic, which flagged 100 %, protects the conclusion.
- **N5 (sensor-correlated error):** the permutation test rejects the true atmospheric null in 70 % of replicates. The fix is the sensor configuration, not the inference.

## 4 · Predictive families on non-Gaussian worlds

Three families, matched within each pair and tuned by day-grouped CV: Gaussian (ridge mean, homoscedastic), location-scale with log-link variance and a KDE residual density mixed 1 % with a wide Gaussian (`ls-kde`), and multinomial logistic over the fixed bins `EDGES` (`binned`). `+T` variants apply a transform rule that never sees V: model log D when D > 0 and the context-only model's held-out score (Jacobian-corrected) prefers it. The contrast is invariant to a shared monotone map, so the target is unchanged. The binned family's estimand is `I(B;V|C)` for the binned depression `B`. By data processing it is at most `I(D;V|C)`, and it is computed exactly here so that the two are never confused.

| world | departure from the Gaussian model | continuous truth |
|---|---|---|
| F1 | none (P1 world) | 0.050 |
| F2 | variance-only information: regime SD 0.7 / 2.5 K revealed by the cue | entropy of a scale mixture |
| F3 | skewed, positive D: `D = 2.5 exp(0.18 (D_lin − 7))`, cue as in P1 with γ = 0.35 | invariant under the monotone map: `I(D_lin;V|C)` |

40 replicates × 60 days per world (binned fitting is ~3 s per model).

In [5]:
def bin_prob(mu, sd, edges, y):
    """Probability of the realised bin of y under N(mu, sd^2), row-wise."""
    e = np.r_[-np.inf, edges, np.inf]
    k = np.searchsorted(edges, y)
    return norm.cdf((e[k + 1] - mu) / sd) - norm.cdf((e[k] - mu) / sd)

MC_DAYS = 10_000  # iid rows: 480 000 per world, Monte-Carlo SE of a truth < 0.002 nats
F1 = DEV['P1']
F2 = dict(spec=gauss_world(0), ctx=E2, vis=['vis_regime'], n_days=60, modify=lambda df, rng: regime((2.5, 0.7))(df, rng).assign(vis_regime=lambda d: d.regime + rng.normal(0, .1, len(d))))
F3 = dict(spec=gauss_world(.35), ctx=E2, vis=['vis_moist'], n_days=60, modify=lambda df, rng: df.assign(d_lin=df.d_ref_k, d_ref_k=2.5 * np.exp(0.18 * (df.d_ref_k - 7))))

def truths(W, kind):
    """(continuous, binned) truths. Continuous: exact (CMI, or scale-mixture quadrature); binned: Monte Carlo on the exact conditionals of an iid copy."""
    df = frame(iid(W), MC_DAYS, 1)
    lat = df.d_lin.to_numpy() if kind == 'lognormal' else df.d_ref_k.to_numpy()
    mc, sc = synthetic.conditional(W['spec'], df, 'd_ref_k', ['temp_cond_c'])
    if kind == 'regime':
        x = np.linspace(-25, 25, 50001)
        mix = .5 * norm.pdf(x, 0, .7) + .5 * norm.pdf(x, 0, 2.5)
        cont = -np.trapezoid(mix * np.log(mix), x) - np.mean([.5 * np.log(2 * np.pi * np.e * s ** 2) for s in (.7, 2.5)])
        sr = np.where(df.regime == 1, .7, 2.5)
        pc = .5 * bin_prob(mc, np.full(len(df), .7), EDGES, lat) + .5 * bin_prob(mc, np.full(len(df), 2.5), EDGES, lat)
        return cont, np.mean(np.log(bin_prob(mc, sr, EDGES, lat)) - np.log(pc))
    mv, sv = synthetic.conditional(W['spec'], df, 'd_ref_k', ['temp_cond_c'] + W['vis'])
    edges = 7 + np.log(np.array(EDGES) / 2.5) / 0.18 if kind == 'lognormal' else EDGES
    return synthetic.cmi(W['spec'], 'd_ref_k', W['vis'], 'temp_cond_c'), np.mean(np.log(bin_prob(mv, sv, edges, lat)) - np.log(bin_prob(mc, sc, edges, lat)))

TRUTH = {'F1': truths(F1, 'gauss'), 'F2': truths(F2, 'regime'), 'F3': truths(F3, 'lognormal')}
pd.DataFrame(TRUTH, index=['continuous', 'binned']).T

,continuous,binned
F1,0.0499,0.0485
F2,0.2274,0.2082
F3,0.1994,0.1865


In [6]:
FAM = {'gauss': {}, 'ls-kde': dict(var='hetero', resid='kde'), 'binned': dict(family='binned', edges=EDGES)}

def log_scale_preferred(C, y, sp, g):
    """Transform rule that never sees V: log D only if D > 0 and the context model's held-out score, Jacobian-corrected, prefers it."""
    return y.min() > 0 and np.nanmean(scoring.crossfit(C, np.log(y), sp, g)[2] - np.log(y)) > np.nanmean(scoring.crossfit(C, y, sp, g)[2])

def fam_rep(W, seed):
    df = frame(W, W['n_days'], seed)
    Xc, Xvc, y, _ = policy.select_sci(df, 'e2', W['ctx'], W['vis'])
    C, X, y, g = Xc.to_numpy(), Xvc.to_numpy(), y.to_numpy(), df.day.to_numpy()
    sp = splits.kfold(splits.assign_folds(g, 5))
    use_log = log_scale_preferred(C, y, sp, g)
    out = {'log_chosen': use_log}
    for k, kw in FAM.items():
        for suffix, yy in [('', y)] + ([('+T', np.log(y) if use_log else y)] if k != 'binned' else []):
            r = scoring.contrast(C, X, yy, sp, g, **kw)
            lo, hi = inference.cluster_interval(r.d, r.groups)
            out |= {k + suffix: r.mean, f'{k}{suffix}_lo': lo, f'{k}{suffix}_hi': hi}
    return out

fam_raw = {k: pd.DataFrame(par()(delayed(fam_rep)(W, s) for s in range(3000, 3040))) for k, W in {'F1': F1, 'F2': F2, 'F3': F3}.items()}

def fam_table(raw, truths):
    rows = []
    for w, a in raw.items():
        for k in ['gauss', 'gauss+T', 'ls-kde', 'ls-kde+T', 'binned']:
            t = truths[w][1] if k == 'binned' else truths[w][0]
            rows.append(dict(world=w, family=k, target='binned' if k == 'binned' else 'continuous', truth=t, mean_est=a[k].mean(), bias=a[k].mean() - t,
                             sd_est=a[k].std(), cover_xt=((a[f'{k}_lo'] <= t) & (a[f'{k}_hi'] >= t)).mean(), rej=(a[f'{k}_lo'] > 0).mean(), log_chosen=a.log_chosen.mean()))
    return pd.DataFrame(rows).set_index(['world', 'family'])

fam_table(fam_raw, TRUTH)

target   truth  mean_est    bias  sd_est  cover_xt    rej  \
world family                                                                    
F1    gauss     continuous  0.0499    0.0509  0.0009  0.0214     0.950  0.800   
      gauss+T   continuous  0.0499    0.0509  0.0009  0.0214     0.950  0.800   
      ls-kde    continuous  0.0499    0.0487 -0.0012  0.0231     0.925  0.675   
      ls-kde+T  continuous  0.0499    0.0487 -0.0012  0.0231     0.925  0.675   
      binned        binned  0.0485    0.0407 -0.0078  0.0241     0.850  0.575   
F2    gauss     continuous  0.2274   -0.0132 -0.2406  0.0170     0.000  0.000   
      gauss+T   continuous  0.2274   -0.0132 -0.2406  0.0170     0.000  0.000   
      ls-kde    continuous  0.2274    0.1886 -0.0388  0.0370     0.800  1.000   
      ls-kde+T  continuous  0.2274    0.1886 -0.0388  0.0370     0.800  1.000   
      binned        binned  0.2082    0.0102 -0.1980  0.0214     0.000  0.050   
F3    gauss     continuous  0.1994    0.1459 -0.0535  0.0314     0.550  0.950   
      gauss+T   continuous  0.1994    0.2036  0.0043  0.0368     0.975  1.000   
      ls-kde    continuous  0.1994    0.1037 -0.0957  0.0357     0.325  0.750   
      ls-kde+T  continuous  0.1994    0.2037  0.0043  0.0386     0.875  1.000   
      binned        binned  0.1865    0.1779 -0.0086  0.0361     0.950  1.000   

                log_chosen  
world family                
F1    gauss            0.0  
      gauss+T          0.0  
      ls-kde           0.0  
      ls-kde+T         0.0  
      binned           0.0  
F2    gauss            0.0  
      gauss+T          0.0  
      ls-kde           0.0  
      ls-kde+T         0.0  
      binned           0.0  
F3    gauss            1.0  
      gauss+T          1.0  
      ls-kde           1.0  
      ls-kde+T         1.0  
      binned           1.0

- **Gaussian:** best on Gaussian data. It fails completely when the information is in the spread (F2: −0.24) and is biased under skew (F3: −0.054) unless the transform rule applies. The rule chose log D in 100 % of F3 samples and 0 % elsewhere, after which the bias is +0.004.
- **`ls-kde`** (1 % defensive component): as efficient as the Gaussian on F1 (SD 0.023 vs 0.021); recovers most of the spread information in F2 (bias −0.039, coverage 0.80); fails on the raw skewed scale (−0.096) and is fine after the transform (+0.004). The first, non-defensive version produced single held-out rows down to −637 nats on F3 (development runs outside the notebook); the wide component bounds them.
- **Binned:** unbiased for its own target on F1 and F3 (−0.008, −0.009); binning itself costs 3–7 % of the continuous information. It is structurally blind to spread-only information around a context-dependent centre (F2: 0.010 vs 0.208). Logits linear in the inputs cannot express a change of scale around a moving mean. V × context interactions recovered 0.148, but cost −0.034 of null bias and dropped F1 to 0.015 (development runs), so they were not adopted.
- **Consequence:** no single family is safe. Exp 2a reports a panel. A GO needs `ls-kde+T`; `gauss+T` is reported for efficiency; binned is used where censoring matters.

## 5 · Censored labels

Near saturation the label is an upper bound. The world is P1 shifted humid (`D ≈ 3 + 4 cos(hour)` K, γ = 0.35), and labels below `c = 1 K` are reported as `≤ 1 K` (left-censored). Four ways of handling them:

- `naive`: fit on the bounds as if exact, score censored rows by the CDF (the M0 code path the review flagged);
- `drop`: discard censored rows (fit and score) — changes the population to `D > c`, i.e. selects on the label;
- `tobit`: penalised censored-Gaussian likelihood for fitting (`scoring.crossfit(..., censored=)`) and CDF scoring;
- `binned`: lowest bin edge at `c`, so every censored row falls exactly in bin 0.

Targets, all exact (Monte Carlo on the true conditionals): `latent` = `I(D;V|C)`, `obs` = information in the censored observation `(max(D, c), 1[D<c])`, `selected` = expected log-score difference of the true truncated conditionals on `D > c` rows, `binned` = `I(B;V|C)` with `EDGES_C`.

In [7]:
CEN = 1.0
EDGES_C = [CEN] + [e for e in EDGES if e > CEN]
def censor(df, rng):
    df['d_lat'] = df.d_ref_k
    df['cens'] = df.d_ref_k < CEN
    df['d_ref_k'] = np.maximum(df.d_ref_k, CEN)
    return df
F4 = dict(spec=gauss_world(.35) | {'d_ref_k': Node({'temp_cond_c': .4, 'dres': 1}, mean=cyc(-5, 2))}, ctx=E2, vis=['vis_moist'], n_days=60, modify=censor)

df4 = frame(iid(F4), MC_DAYS, 1)
mc, sc = synthetic.conditional(F4['spec'], df4, 'd_ref_k', ['temp_cond_c'])
mv, sv = synthetic.conditional(F4['spec'], df4, 'd_ref_k', ['temp_cond_c', 'vis_moist'])
y4, c4, lat = df4.d_ref_k.to_numpy(), df4.cens.to_numpy(), df4.d_lat.to_numpy()
keep = ~c4
trunc = lambda m, s: scoring.gauss_ls(y4[keep], m[keep], s[keep]) - norm.logsf(CEN, m[keep], s[keep])
T4 = {'latent': synthetic.cmi(F4['spec'], 'd_ref_k', 'vis_moist', 'temp_cond_c'),
      'obs': np.mean(scoring.gauss_ls(y4, mv, sv, c4) - scoring.gauss_ls(y4, mc, sc, c4)),
      'selected': np.mean(trunc(mv, sv) - trunc(mc, sc)),
      'binned': np.mean(np.log(bin_prob(mv, sv, EDGES_C, lat)) - np.log(bin_prob(mc, sc, EDGES_C, lat)))}
print(f'censored share {c4.mean():.1%};', {k: round(v, 4) for k, v in T4.items()})

censored share 13.5%; {'latent': np.float64(0.1994), 'obs': np.float64(0.1878), 'selected': np.float64(0.1716), 'binned': np.float64(0.1836)}


In [8]:
def cen_rep(W, seed):
    df = frame(W, W['n_days'], seed)
    Xc, Xvc, y, _ = policy.select_sci(df, 'e2', W['ctx'], W['vis'])
    C, X, y, g, cens = Xc.to_numpy(), Xvc.to_numpy(), y.to_numpy(), df.day.to_numpy(), df.cens.to_numpy()
    sp = splits.kfold(splits.assign_folds(g, 5))
    naive = [scoring.crossfit(M, y, sp, g) for M in (C, X)]
    out = {'naive': np.mean(scoring.gauss_ls(y, *naive[1][:2], cens) - scoring.gauss_ls(y, *naive[0][:2], cens)),
           'tobit': scoring.contrast(C, X, y, sp, g, censored=cens).mean,
           'binned': scoring.contrast(C, X, y, sp, g, family='binned', edges=W.get('edges', EDGES_C), censored=cens).mean}
    k = ~cens
    out['drop'] = scoring.contrast(C[k], X[k], y[k], splits.kfold(splits.assign_folds(g[k], 5)), g[k]).mean
    return out

cen = pd.DataFrame(par()(delayed(cen_rep)(F4, s) for s in range(4000, 4040)))
target = {'naive': 'obs', 'tobit': 'obs', 'binned': 'binned', 'drop': 'selected'}
pd.DataFrame({k: {'target': target[k], 'truth': T4[target[k]], 'mean_est': cen[k].mean(), 'bias': cen[k].mean() - T4[target[k]], 'sd_est': cen[k].std(),
                  'vs_latent': cen[k].mean() - T4['latent']} for k in cen}).T

,target,truth,mean_est,bias,sd_est,vs_latent
naive,obs,0.1878,0.1918,0.004,0.0348,-0.0076
tobit,obs,0.1878,0.1992,0.0114,0.0357,-0.0002
binned,binned,0.1836,0.1882,0.0045,0.0382,-0.0112
drop,selected,0.1716,0.1843,0.0127,0.0363,-0.0151


- **Truth computation, corrected:** the first version of this notebook computed Monte-Carlo truths on one 2000-day *dependent* sample. With 62–66 % of residual variance between days, that sample's realised covariance was 7 % below its expectation, and the "truth" was off by 0.02 nats, which made every method look biased. Truths now come from iid copies of each world (same marginal distribution, 480 000 rows).
- **Tobit** targets the observation-level information and lands +0.011 above it (≈ 2 SE of the mean); naive fitting lands +0.004, binned +0.0045 against its binned target. With 13.5 % censoring the M0 bug (bounds fitted as values) had little effect in *this* world. Nothing guarantees that in fog-heavy regimes, so Tobit (or binned) is the required likelihood.
- **Dropping censored rows** estimates a different quantity: the selected-population contrast (truth 0.172), not `I(D;V|C)` (0.199) or the observation-level information (0.188). Selecting on the label is not an acceptable shortcut for confirmatory work.

## 6 · Held-out validation suite

**Procedures fixed by the development suite** (decision addendum, `docs/m0_decision_addendum.md`):

- Q0, zero information: refitting day-block permutation test (Gaussian statistic, residuals re-projected on the context's orthogonal complement), 39 draws, p ≤ 0.05.
- Q1, effect size: frozen hold-out (half of the days) with a day-cluster t interval; its target is the information captured by the frozen models.
- Adequacy: the context-expansion diagnostic is mandatory before any contrast is interpreted.
- Families: Gaussian with the transform rule (efficient baseline), `ls-kde` with the transform rule (robust), binned (exact under censoring). Tobit for censored Gaussian fits.

**Acceptance criteria, declared before the first validation run.** 150 replicates for the inference worlds (Monte-Carlo SE of a 0.05 rate ≈ 0.018), 40 for the family worlds.

| id | criterion |
|---|---|
| A1 | permutation Type I ≤ 0.085 in V-N1 (well-specified), V-N2 (multimodal + strong serial dependence), V-N3 (hour-misspecified context) |
| A2 | permutation power ≥ 0.80 at truth 0.05 (V-P1, 90 days) |
| A3 | frozen 95 % interval covers its frozen target ≥ 0.90 and the truth ≥ 0.88 in V-P1 and V-P2 |
| A4 | cross-fitted Gaussian |bias| ≤ 0.01 in V-P1 and V-P2 |
| A5 | context diagnostic flags the temperature-misspecified null V-N4 in ≥ 90 % of replicates |
| A6 | `ls-kde+T` |bias| ≤ 0.04 in V-F1 (regime) and V-F2 (skewed), ≤ 0.01 in V-P1; binned |bias| ≤ 0.02 (against binned truth) in V-F2 and V-F3 |
| A7 | Tobit |bias| ≤ 0.015 against the observation-level truth in V-F3 (censored) |
| — | V-S (shared 1.5 K radiation error): rejection rate of the true atmospheric null reported, no criterion; no inference procedure can repair it |

Validation worlds: 36 samples/day over 07–19 h (20 min), 90 days, β = 0.6, T anomaly SD 4 K, moisture residual SD 1.5 K (60 % day-level, AR 0.85), visual noise 50 % day-level (AR 0.7). Seeds ≥ 90 000. None of these settings was used in development.

In [9]:
VP = dict(per_day=36, hours=(7, 19), n_days=90)
vw = lambda gamma: gauss_world(gamma, beta=.6, t_sd=4, r_sd=1.5, r_day=.6, r_ar=.85, v_day=.5, v_ar=.7)
g05, g10 = np.sqrt(np.expm1(.1)) / 1.5, np.sqrt(np.expm1(.2)) / 1.5

def bump(df, rng):
    df['d_ref_k'] = df.d_ref_k + 2 * np.exp(-(df.temp_cond_c - 24) ** 2 / 8)
    df['vis_bump'] = 2 * np.exp(-(df.temp_cond_c - 24) ** 2 / 8) + rng.normal(0, .5, len(df))
    return df

sun2 = {'vis_sun': Node(sd=.05, mean=lambda h: np.cos(2 * np.pi * (h - 11) / 24))}
VAL = {'V-N1': VP | dict(spec=vw(0), ctx=E2, vis=['vis_moist'], truth=0.0),
       'V-N2': VP | dict(spec=gauss_world(0, beta=.6, t_sd=4, r_sd=1.5, r_day=.85, r_ar=.97, v_day=.5, v_ar=.7), ctx=E2, vis=['vis_moist'], truth=0.0, modify=regime((2.0, 0.5), .3)),
       'V-N3': VP | dict(spec=vw(0) | sun2, ctx=['hour_local', 'temp_cond_c'], vis=['vis_sun'], truth=0.0),
       'V-N4': VP | dict(spec=vw(0), ctx=E2, vis=['vis_bump'], truth=0.0, modify=bump),
       'V-S': VP | dict(spec=rad_world(1.5), ctx=E2, vis=['vis_bright'], truth=synthetic.cmi(rad_world(1.5), 'd_ref_k', 'vis_bright', 'temp_cond_c'), shared=True),
       'V-P1': VP | dict(spec=vw(g05), ctx=E2, vis=['vis_moist'], truth=synthetic.cmi(vw(g05), 'd_ref_k', 'vis_moist', 'temp_cond_c')),
       'V-P2': VP | dict(spec=vw(g10), ctx=E2, vis=['vis_moist'], truth=synthetic.cmi(vw(g10), 'd_ref_k', 'vis_moist', 'temp_cond_c'))}
val_raw = {k: run(W, range(90000, 90150)) for k, W in VAL.items()}
val = pd.DataFrame([summarise(k, VAL[k], a) for k, a in val_raw.items()]).set_index('world')
val[['truth', 'bias', 'sd_est', 'rej_perm', 'rej_frz', 'rej_xt', 'cov_frz', 'cov_frz_target', 'cov_xt', 'diag_flag']]

,truth,bias,sd_est,rej_perm,rej_frz,rej_xt,cov_frz,cov_frz_target,cov_xt,diag_flag
world,,,,,,,,,,
V-N1,0.0000,-0.0022,0.0055,0.0600,0.0400,0.0000,0.9133,0.8933,0.8533,0.0067
V-N2,0.0000,-0.0036,0.0073,0.0467,0.0467,0.0067,0.8733,0.9000,0.8467,0.0067
V-N3,0.0000,0.0105,0.0052,0.0000,0.4467,0.6467,0.6667,0.9600,0.5133,0.1933
V-N4,0.0000,0.0616,0.0159,1.0000,0.8267,1.0000,0.2133,0.9333,0.0000,0.3867
V-S,0.0669,-0.0040,0.0276,0.9933,0.4533,0.7733,0.9200,0.9267,0.9067,0.0000
V-P1,0.0500,-0.0029,0.0247,0.9733,0.4867,0.7667,0.9267,0.9267,0.8933,0.0067
V-P2,0.1000,-0.0031,0.0337,1.0000,0.7667,0.9667,0.9400,0.9467,0.9333,0.0067


In [10]:
def regime_truth(sds, p):
    x = np.linspace(-25, 25, 50001)
    mix = (1 - p) * norm.pdf(x, 0, sds[0]) + p * norm.pdf(x, 0, sds[1])
    return -np.trapezoid(mix * np.log(mix), x) - ((1 - p) * .5 * np.log(2 * np.pi * np.e * sds[0] ** 2) + p * .5 * np.log(2 * np.pi * np.e * sds[1] ** 2))

VF1 = VP | dict(spec=vw(0), ctx=E2, vis=['vis_regime'], modify=lambda df, rng: regime((2.0, 0.6), .4)(df, rng).assign(vis_regime=lambda d: d.regime + rng.normal(0, .1, len(d))))
VF2 = VP | dict(spec=vw(g10), ctx=E2, vis=['vis_moist'], modify=lambda df, rng: df.assign(d_lin=df.d_ref_k, d_ref_k=3 * np.exp(0.15 * (df.d_ref_k - 7))))
CEN_V = 0.8
EDGES_CV = [CEN_V] + [e for e in EDGES if e > CEN_V]
VF3 = VP | dict(spec=vw(g10) | {'d_ref_k': Node({'temp_cond_c': .6, 'dres': 1}, mean=cyc(-9, 1))}, ctx=E2, vis=['vis_moist'],
                modify=lambda df, rng: df.assign(d_lat=df.d_ref_k, cens=df.d_ref_k < CEN_V, d_ref_k=np.maximum(df.d_ref_k, CEN_V)))

df = frame(iid(VF2), MC_DAYS, 1)
mc, sc = synthetic.conditional(VF2['spec'], df, 'd_ref_k', ['temp_cond_c'])
mv, sv = synthetic.conditional(VF2['spec'], df, 'd_ref_k', ['temp_cond_c', 'vis_moist'])
ze = 7 + np.log(np.array(EDGES) / 3) / 0.15
d1 = frame(iid(VF1), MC_DAYS, 1)
m1, _ = synthetic.conditional(VF1['spec'], d1, 'd_ref_k', ['temp_cond_c'])
y1, s1 = d1.d_ref_k.to_numpy(), np.where(d1.regime == 1, .6, 2.0)
pc1 = .6 * bin_prob(m1, np.full(len(d1), 2.0), EDGES, y1) + .4 * bin_prob(m1, np.full(len(d1), .6), EDGES, y1)
VTRUTH = {'V-F1': (regime_truth((2.0, 0.6), .4), np.mean(np.log(bin_prob(m1, s1, EDGES, y1)) - np.log(pc1))), 'V-F2': (synthetic.cmi(VF2['spec'], 'd_ref_k', 'vis_moist', 'temp_cond_c'), np.mean(np.log(bin_prob(mv, sv, ze, df.d_lin)) - np.log(bin_prob(mc, sc, ze, df.d_lin))))}
df = frame(iid(VF3), MC_DAYS, 1)
mc, sc = synthetic.conditional(VF3['spec'], df, 'd_ref_k', ['temp_cond_c'])
mv, sv = synthetic.conditional(VF3['spec'], df, 'd_ref_k', ['temp_cond_c', 'vis_moist'])
y, c, lat = df.d_ref_k.to_numpy(), df.cens.to_numpy(), df.d_lat.to_numpy()
VT3 = {'latent': synthetic.cmi(VF3['spec'], 'd_ref_k', 'vis_moist', 'temp_cond_c'), 'obs': np.mean(scoring.gauss_ls(y, mv, sv, c) - scoring.gauss_ls(y, mc, sc, c)),
       'binned': np.mean(np.log(bin_prob(mv, sv, EDGES_CV, lat)) - np.log(bin_prob(mc, sc, EDGES_CV, lat)))}
print(f'V-F3 censored share {c.mean():.1%};', {k: round(v, 4) for k, v in VT3.items()})

vfam_raw = {k: pd.DataFrame(par()(delayed(fam_rep)(W, s) for s in range(91000, 91040))) for k, W in {'V-F1': VF1, 'V-F2': VF2}.items()}
vfam = fam_table(vfam_raw, VTRUTH)
vcen = pd.DataFrame(par()(delayed(cen_rep)(VF3 | dict(edges=EDGES_CV), s) for s in range(92000, 92040)))
display(vfam)
pd.DataFrame({k: {'truth': VT3[t], 'mean_est': vcen[k].mean(), 'bias': vcen[k].mean() - VT3[t], 'sd_est': vcen[k].std()} for k, t in [('tobit', 'obs'), ('binned', 'binned'), ('naive', 'obs')]}).T

V-F3 censored share 10.4%; {'latent': np.float64(0.1), 'obs': np.float64(0.0949), 'binned': np.float64(0.0917)}


target   truth    mean_est    bias  sd_est  cover_xt  \
world family                                                               
V-F1  gauss     continuous  0.1948 -2.6084e-03 -0.1974  0.0068     0.000   
      gauss+T   continuous  0.1948 -2.6084e-03 -0.1974  0.0068     0.000   
      ls-kde    continuous  0.1948  1.6278e-01 -0.0320  0.0316     0.700   
      ls-kde+T  continuous  0.1948  1.6278e-01 -0.0320  0.0316     0.700   
      binned        binned  0.1634  3.3465e-05 -0.1633  0.0132     0.000   
V-F2  gauss     continuous  0.1000  5.7339e-02 -0.0427  0.0242     0.550   
      gauss+T   continuous  0.1000  9.6796e-02 -0.0032  0.0236     0.950   
      ls-kde    continuous  0.1000  5.2031e-02 -0.0480  0.0196     0.625   
      ls-kde+T  continuous  0.1000  9.4918e-02 -0.0051  0.0223     0.925   
      binned        binned  0.0887  7.5353e-02 -0.0134  0.0236     0.925   

                  rej  log_chosen  
world family                       
V-F1  gauss     0.000         0.0  
      gauss+T   0.000         0.0  
      ls-kde    1.000         0.0  
      ls-kde+T  1.000         0.0  
      binned    0.000         0.0  
V-F2  gauss     0.750         1.0  
      gauss+T   1.000         1.0  
      ls-kde    0.500         1.0  
      ls-kde+T  0.975         1.0  
      binned    0.875         1.0

,truth,mean_est,bias,sd_est
tobit,0.0949,0.0955,0.0006,0.0206
binned,0.0917,0.0862,-0.0055,0.0203
naive,0.0949,0.0898,-0.0051,0.0195


In [11]:
vp1 = fam_table({'V-P1': pd.DataFrame(par()(delayed(fam_rep)(VAL['V-P1'], s) for s in range(93000, 93040)))}, {'V-P1': (VAL['V-P1']['truth'], np.nan)})
v = lambda w, col: val.loc[w, col]
acc = [('A1', 'perm Type I <= 0.085: V-N1 / V-N2 / V-N3', [v(w, 'rej_perm') for w in ('V-N1', 'V-N2', 'V-N3')], all(v(w, 'rej_perm') <= .085 for w in ('V-N1', 'V-N2', 'V-N3'))),
       ('A2', 'perm power >= 0.80 (V-P1)', v('V-P1', 'rej_perm'), v('V-P1', 'rej_perm') >= .8),
       ('A3', 'frozen covers target >= 0.90 and truth >= 0.88 (V-P1, V-P2)', [v(w, c) for w in ('V-P1', 'V-P2') for c in ('cov_frz_target', 'cov_frz')],
        all(v(w, 'cov_frz_target') >= .9 and v(w, 'cov_frz') >= .88 for w in ('V-P1', 'V-P2'))),
       ('A4', 'cross-fit |bias| <= 0.01 (V-P1, V-P2)', [v(w, 'bias') for w in ('V-P1', 'V-P2')], all(abs(v(w, 'bias')) <= .01 for w in ('V-P1', 'V-P2'))),
       ('A5', 'diagnostic flags V-N4 >= 0.90', v('V-N4', 'diag_flag'), v('V-N4', 'diag_flag') >= .9),
       ('A6', 'ls-kde+T |bias| <= 0.04 (V-F1, V-F2), <= 0.01 (V-P1)', [vfam.loc[('V-F1', 'ls-kde+T'), 'bias'], vfam.loc[('V-F2', 'ls-kde+T'), 'bias'], vp1.loc[('V-P1', 'ls-kde+T'), 'bias']],
        abs(vfam.loc[('V-F1', 'ls-kde+T'), 'bias']) <= .04 and abs(vfam.loc[('V-F2', 'ls-kde+T'), 'bias']) <= .04 and abs(vp1.loc[('V-P1', 'ls-kde+T'), 'bias']) <= .01),
       ('A6', 'binned |bias| <= 0.02 (V-F2, V-F3)', [vfam.loc[('V-F2', 'binned'), 'bias'], vcen.binned.mean() - VT3['binned']],
        abs(vfam.loc[('V-F2', 'binned'), 'bias']) <= .02 and abs(vcen.binned.mean() - VT3['binned']) <= .02),
       ('A7', 'Tobit |bias| <= 0.015 vs observation-level truth (V-F3)', vcen.tobit.mean() - VT3['obs'], abs(vcen.tobit.mean() - VT3['obs']) <= .015)]
pd.DataFrame([(i, c, np.round(x, 4), p) for i, c, x, p in acc], columns=['id', 'criterion', 'value', 'pass'])

,id,criterion,value,pass
0,A1,perm Type I <= 0.085: V-N1 / V-N2 / V-N3,"[0.06, 0.0467, 0.0]",True
1,A2,perm power >= 0.80 (V-P1),0.9733,True
2,A3,frozen covers target >= 0.90 and truth >= 0.88...,"[0.9267, 0.9267, 0.9467, 0.94]",True
3,A4,"cross-fit |bias| <= 0.01 (V-P1, V-P2)","[-0.0029, -0.0031]",True
4,A5,diagnostic flags V-N4 >= 0.90,0.3867,False
5,A6,"ls-kde+T |bias| <= 0.04 (V-F1, V-F2), <= 0.01 ...","[-0.032, -0.0051, -0.0025]",True
6,A6,"binned |bias| <= 0.02 (V-F2, V-F3)","[-0.0134, -0.0055]",True
7,A7,Tobit |bias| <= 0.015 vs observation-level tru...,0.0006,True


**Validation outcome: 7 of 8 criteria pass; A5 fails.** In V-N4 (true information 0; context linear in T while D has a temperature "bump" that the cue encodes), the polynomial adequacy diagnostic flagged only 39 % of replicates, while the permutation test rejected the null in 100 % and the frozen test in 83 %, at a spurious 0.062 nats. This is the N3b mechanism with a non-linearity outside the squares/products expansion. **The revised estimator therefore does not pass all of its pre-declared acceptance tests.**

Passed on worlds not used in development: permutation validity on well-specified, multimodal + strongly serial, and hour-misspecified nulls (0.060 / 0.047 / 0.000); power 0.97 at 0.05 nats and 90 days; frozen coverage of its target 0.93–0.95 and of the truth 0.93–0.94; cross-fitted bias −0.003; `ls-kde+T` within 0.032 on regime and skewed worlds and 0.003 on Gaussian; binned within 0.013; Tobit within 0.001. As expected, sensor-correlated error (V-S) leads every powerful test to reject the true atmospheric null (permutation 99 %).

**Next, not done here:** a context model whose capacity in T is flexible (spline or tree basis in T and hour × T), given identically to the joint model, plus a non-parametric adequacy diagnostic. Develop it on the development worlds and validate on a third, fresh suite (seeds ≥ 100 000), not on V-N4 again.